# Notebook 05: ML Forecasting - Prophet Time Series

## Overview
This notebook trains Prophet models to forecast cholera cases 4 weeks ahead with uncertainty intervals.

## Prerequisites
- Notebook 04 completed successfully
- Gold tables: `epi_analytics_weekly`, `epi_country_trends`
- Prophet library installed (or mock_prophet for testing)

## Inputs
- Historical case data from Gold layer

## Outputs
- `gold.forecast_cases_4wk` - 4-week ahead forecasts with uncertainty
- `gold.model_performance` - Model evaluation metrics

## Execution Time
~2-3 minutes

In [1]:
# ============================================
# ENVIRONMENT DETECTION & CONFIGURATION
# ============================================

import os
import sys
from pathlib import Path

IS_FABRIC = os.path.exists('/lakehouse/default')

if IS_FABRIC:
    print("🌐 Running in Microsoft Fabric")
    GOLD_TABLE_PATH = "/lakehouse/default/Tables/gold"
else:
    print("💻 Running locally")
    project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(project_root / 'src'))
    GOLD_TABLE_PATH = str(project_root / "data" / "gold_tables")

print(f"Gold Path: {GOLD_TABLE_PATH}")

💻 Running locally
Gold Path: D:\Projects\cholera-cdr-mvp\data\gold_tables


In [2]:
# ============================================
# IMPORTS
# ============================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import logging

# Try to import Prophet, fall back to mock if not available
try:
    from prophet import Prophet
    USE_REAL_PROPHET = True
    print("✅ Using real Prophet library")
except ImportError:
    try:
        from epi_analytics.mock_prophet import Prophet
        USE_REAL_PROPHET = False
        print("⚠️  Using mock Prophet (for testing without Stan backend)")
    except ImportError:
        print("❌ Neither Prophet nor mock_prophet available")
        raise

# Import forecasting functions
try:
    from epi_analytics.forecasting import train_prophet_model, explain_forecast
    print("✅ Imported forecasting functions")
except ImportError:
    print("⚠️  Forecasting module not available, using inline functions")
    train_prophet_model = None
    explain_forecast = None

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Imports successful")

Importing plotly failed. Interactive plots will not work.


✅ Using real Prophet library
✅ Imported forecasting functions
✅ Imports successful


In [3]:
# ============================================
# LOAD GOLD DATA
# ============================================

print("\n📂 Loading Gold layer data...\n")

try:
    if IS_FABRIC:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        df_weekly = spark.table("gold.epi_analytics_weekly").toPandas()
        df_country_trends = spark.table("gold.epi_country_trends").toPandas()
    else:
        df_weekly = pd.read_parquet(Path(GOLD_TABLE_PATH) / "epi_analytics_weekly.parquet")
        df_country_trends = pd.read_parquet(Path(GOLD_TABLE_PATH) / "epi_country_trends.parquet")
    
    print(f"✅ Loaded {len(df_weekly)} weekly records")
    print(f"✅ Loaded {len(df_country_trends)} country trend records")
    
except Exception as e:
    logger.error(f"Error loading Gold data: {e}")
    raise


📂 Loading Gold layer data...

✅ Loaded 3 weekly records
✅ Loaded 6 country trend records


In [5]:
# ============================================
# STEP 3: PREPARE FORECASTING DATA
# ============================================

print("\n" + "="*60)
print("🔄 PREPARING FORECASTING DATASET")
print("="*60 + "\n")

# Check what columns we have
print("Available columns in df_weekly:")
print(df_weekly.columns.tolist())
print()

# Create date column from epi_year and epi_week if it doesn't exist
if 'date' not in df_weekly.columns:
    print("Creating 'date' column from epi_year and epi_week...")
    
    from datetime import timedelta
    
    def epi_week_to_date(row):
        """Convert epi_year and epi_week to date (Sunday of that week)"""
        year_start = pd.Timestamp(year=int(row['epi_year']), month=1, day=1)
        days_to_sunday = (6 - year_start.dayofweek) % 7
        first_sunday = year_start + timedelta(days=days_to_sunday)
        week_date = first_sunday + timedelta(weeks=int(row['epi_week']) - 1)
        return week_date
    
    df_weekly['date'] = df_weekly.apply(epi_week_to_date, axis=1)
    print(f"✅ Created date column\n")

# Prepare for Prophet (requires 'ds' and 'y')
df_forecast_input = df_weekly[['date', 'new_cases']].copy()
df_forecast_input.columns = ['ds', 'y']
df_forecast_input['ds'] = pd.to_datetime(df_forecast_input['ds'])
df_forecast_input = df_forecast_input.sort_values('ds').reset_index(drop=True)
df_forecast_input = df_forecast_input.dropna()

print(f"✅ Forecasting dataset prepared:")
print(f"   - Records: {len(df_forecast_input)}")
print(f"   - Date range: {df_forecast_input['ds'].min().date()} to {df_forecast_input['ds'].max().date()}")
print(f"   - Total cases: {df_forecast_input['y'].sum():.0f}")
print(f"   - Average: {df_forecast_input['y'].mean():.0f} cases/week")

print("\n📋 Sample data:")
print(df_forecast_input)

# Data sufficiency check
MIN_RECOMMENDED_WEEKS = 26
weeks_available = len(df_forecast_input)

print(f"\n📊 Data sufficiency check:")
if weeks_available < MIN_RECOMMENDED_WEEKS:
    print(f"   ⚠️ Limited data: {weeks_available} weeks (recommended: 26+)")
    print(f"   Forecast will be for demonstration only")
    use_seasonality = False
else:
    print(f"   ✅ Sufficient data: {weeks_available} weeks")
    use_seasonality = True

print()


🔄 PREPARING FORECASTING DATASET

Available columns in df_weekly:
['epi_year', 'epi_week', 'new_cases', 'new_deaths', 'affected_countries', 'weekly_cfr', 'anomaly_count']

Creating 'date' column from epi_year and epi_week...
✅ Created date column

✅ Forecasting dataset prepared:
   - Records: 3
   - Date range: 2025-02-09 to 2025-02-23
   - Total cases: 2053
   - Average: 684 cases/week

📋 Sample data:
          ds    y
0 2025-02-09  600
1 2025-02-16  781
2 2025-02-23  672

📊 Data sufficiency check:
   ⚠️ Limited data: 3 weeks (recommended: 26+)
   Forecast will be for demonstration only



In [7]:
# ============================================
# DATA SUFFICIENCY CHECK
# ============================================

print("\n📊 Checking data sufficiency for forecasting...\n")

MIN_RECOMMENDED_WEEKS = 26  # 6 months
IDEAL_WEEKS = 52  # 1 year

weeks_available = len(df_forecast_input)

if weeks_available < MIN_RECOMMENDED_WEEKS:
    print(f"⚠️ LIMITED DATA WARNING:")
    print(f"   Available: {weeks_available} weeks")
    print(f"   Recommended: {MIN_RECOMMENDED_WEEKS}+ weeks")
    print(f"   Ideal: {IDEAL_WEEKS}+ weeks")
    print()
    print("   Impact on forecast:")
    print("   - No seasonal patterns detectable")
    print("   - Wide prediction intervals")
    print("   - Limited trend confidence")
    print("   - Use for demonstration only")
    print()
    
    # Disable seasonality for limited data
    use_seasonality = False
else:
    use_seasonality = True
    print(f"✅ Sufficient data: {weeks_available} weeks")
    print()


📊 Checking data sufficiency for forecasting...

⚠️ LIMITED DATA WARNING:
   Available: 3 weeks
   Recommended: 26+ weeks
   Ideal: 52+ weeks

   Impact on forecast:
   - No seasonal patterns detectable
   - Wide prediction intervals
   - Limited trend confidence
   - Use for demonstration only



In [14]:
# ============================================
# STEP 4: CREATE SIMPLE TREND FORECAST
# ============================================

print("\n" + "="*60)
print("CREATING FORECAST MODEL")
print("="*60 + "\n")

print("⚠️ Note: Using simple linear trend model due to:")
print("   1. Limited historical data (3 weeks)")
print("   2. Prophet backend not fully configured")
print("   This is sufficient for MVP demonstration")
print()

# Calculate simple linear trend from available data
from scipy import stats

# Prepare data for trend calculation
df_forecast_input['x'] = range(len(df_forecast_input))
slope, intercept, r_value, p_value, std_err = stats.linregress(
    df_forecast_input['x'], 
    df_forecast_input['y']
)

print(f"📊 Trend Analysis:")
print(f"   - Slope: {slope:.2f} cases/week")
print(f"   - R²: {r_value**2:.3f}")
print(f"   - Direction: {'INCREASING' if slope > 0 else 'DECREASING'}")
print()

# Calculate mean and std for prediction intervals
mean_cases = df_forecast_input['y'].mean()
std_cases = df_forecast_input['y'].std()

print(f"📊 Historical Statistics:")
print(f"   - Mean: {mean_cases:.0f} cases/week")
print(f"   - Std Dev: {std_cases:.0f}")
print()

print("✅ Forecast model prepared!")
print()


# ============================================
# STEP 5: GENERATE 4-WEEK FORECAST
# ============================================

print("\n" + "="*60)
print("📈 GENERATING 4-WEEK FORECAST")
print("="*60 + "\n")

# Generate forecast dates (4 weeks ahead)
last_date = df_forecast_input['ds'].max()
forecast_dates = []
for i in range(1, 5):  # 4 weeks
    forecast_date = last_date + pd.Timedelta(weeks=i)
    forecast_dates.append(forecast_date)

print(f"Forecasting for weeks after {last_date.date()}:")
for i, date in enumerate(forecast_dates, 1):
    print(f"   Week {i}: {date.date()}")
print()

# Create forecast dataframe
df_forecast = pd.DataFrame({
    'forecast_date': forecast_dates
})

# Calculate forecasted values using linear trend
df_forecast['weeks_ahead'] = range(1, 5)
df_forecast['predicted_cases'] = [
    intercept + slope * (len(df_forecast_input) + i) 
    for i in df_forecast['weeks_ahead']
]

# Ensure non-negative predictions
df_forecast['predicted_cases'] = df_forecast['predicted_cases'].clip(lower=0)

# Calculate prediction intervals (80% and 95%)
# Use expanding uncertainty based on weeks ahead
for idx, row in df_forecast.iterrows():
    weeks_ahead = row['weeks_ahead']
    uncertainty_multiplier = 1 + (weeks_ahead * 0.3)  # Increase uncertainty over time
    
    # 80% interval (±1.28 std dev)
    df_forecast.loc[idx, 'lower_80'] = max(0, row['predicted_cases'] - 1.28 * std_cases * uncertainty_multiplier)
    df_forecast.loc[idx, 'upper_80'] = row['predicted_cases'] + 1.28 * std_cases * uncertainty_multiplier
    
    # 95% interval (±1.96 std dev)
    df_forecast.loc[idx, 'lower_95'] = max(0, row['predicted_cases'] - 1.96 * std_cases * uncertainty_multiplier)
    df_forecast.loc[idx, 'upper_95'] = row['predicted_cases'] + 1.96 * std_cases * uncertainty_multiplier

# Add metadata
df_forecast['confidence_level_80'] = 0.80
df_forecast['confidence_level_95'] = 0.95
df_forecast['model_type'] = 'Linear Trend'
df_forecast['forecast_generated_at'] = datetime.now()

# Round predictions
df_forecast['predicted_cases'] = df_forecast['predicted_cases'].round(0)
df_forecast['lower_80'] = df_forecast['lower_80'].round(0)
df_forecast['upper_80'] = df_forecast['upper_80'].round(0)
df_forecast['lower_95'] = df_forecast['lower_95'].round(0)
df_forecast['upper_95'] = df_forecast['upper_95'].round(0)

print("✅ 4-week forecast generated!")
print()

print("📋 Forecast Summary:")
print(df_forecast[[
    'forecast_date', 'weeks_ahead', 'predicted_cases', 
    'lower_80', 'upper_80', 'lower_95', 'upper_95'
]].to_string(index=False))
print()

# Summary statistics
print("📊 Forecast Statistics:")
print(f"   - Average predicted cases: {df_forecast['predicted_cases'].mean():.0f}/week")
print(f"   - Trend: {slope:.1f} cases/week")
print(f"   - Total predicted (4 weeks): {df_forecast['predicted_cases'].sum():.0f} cases")
print()


CREATING FORECAST MODEL

⚠️ Note: Using simple linear trend model due to:
   1. Limited historical data (3 weeks)
   2. Prophet backend not fully configured
   This is sufficient for MVP demonstration

📊 Trend Analysis:
   - Slope: 36.00 cases/week
   - R²: 0.156
   - Direction: INCREASING

📊 Historical Statistics:
   - Mean: 684 cases/week
   - Std Dev: 91

✅ Forecast model prepared!


📈 GENERATING 4-WEEK FORECAST

Forecasting for weeks after 2025-02-23:
   Week 1: 2025-03-02
   Week 2: 2025-03-09
   Week 3: 2025-03-16
   Week 4: 2025-03-23

✅ 4-week forecast generated!

📋 Forecast Summary:
forecast_date  weeks_ahead  predicted_cases  lower_80  upper_80  lower_95  upper_95
   2025-03-02            1            792.0     641.0     944.0     560.0    1025.0
   2025-03-09            2            828.0     642.0    1015.0     543.0    1114.0
   2025-03-16            3            864.0     643.0    1086.0     525.0    1204.0
   2025-03-23            4            900.0     644.0    1157.0 

In [15]:
# ============================================
# STEP 4: TRAIN PROPHET MODEL
# ============================================

print("\n" + "="*60)
print("🤖 TRAINING PROPHET MODEL")
print("="*60 + "\n")

# Configure Prophet for limited data
print("Configuring Prophet model...")

if weeks_available < MIN_RECOMMENDED_WEEKS:
    # Simplified model for limited data
    model = Prophet(
        yearly_seasonality=False,   # Not enough data
        weekly_seasonality=False,   # Not enough data
        daily_seasonality=False,    # Not relevant for weekly data
        changepoint_prior_scale=0.05,  # Less sensitive to noise
        interval_width=0.95,        # Wider prediction intervals
        n_changepoints=2            # Fewer changepoints for limited data
    )
    print("   - Using simplified model (limited data)")
    print("   - Seasonality: Disabled")
    print("   - Changepoints: 2")
else:
    # Full model with seasonality
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,   # Not relevant for weekly aggregated data
        daily_seasonality=False,
        changepoint_prior_scale=0.1,
        interval_width=0.95,
        n_changepoints=10
    )
    print("   - Using full model")
    print("   - Yearly seasonality: Enabled")

print()

# Train the model
print("Training Prophet model...")
try:
    model.fit(df_forecast_input)
    print("✅ Model training complete!")
except Exception as e:
    print(f"❌ Model training failed: {e}")
    print("   This may be due to insufficient data")
    raise

print()


🤖 TRAINING PROPHET MODEL

Configuring Prophet model...


AttributeError: 'Prophet' object has no attribute 'stan_backend'

In [18]:
# ============================================
# STEP 7: PREPARE FORECAST OUTPUT
# ============================================

print("\n" + "="*60)
print("📈 PREPARING FORECAST OUTPUT")
print("="*60 + "\n")

# The forecast is already prepared in df_forecast
# Just need to format it for Gold layer output

df_forecast_output = df_forecast.copy()

# Rename columns for consistency with expected schema
df_forecast_output = df_forecast_output.rename(columns={
    'forecast_date': 'forecast_week_end_date',
    'predicted_cases': 'predicted_cases',
    'lower_95': 'predicted_cases_lower',
    'upper_95': 'predicted_cases_upper'
})

# Add metadata columns
df_forecast_output['confidence_level'] = 95
df_forecast_output['forecast_generated_date'] = datetime.now().date()
df_forecast_output['forecast_horizon_weeks'] = 4
df_forecast_output['model_type'] = 'Linear Trend'
df_forecast_output['model_version'] = '1.0_mvp'
df_forecast_output['data_quality_note'] = 'Limited historical data (3 weeks) - demonstration only'

# Ensure all numeric columns are integers
numeric_cols = ['predicted_cases', 'predicted_cases_lower', 'predicted_cases_upper', 'lower_80', 'upper_80']
for col in numeric_cols:
    if col in df_forecast_output.columns:
        df_forecast_output[col] = df_forecast_output[col].astype(int)

# Select final columns
output_cols = [
    'forecast_week_end_date', 'weeks_ahead', 'predicted_cases', 
    'predicted_cases_lower', 'predicted_cases_upper',
    'lower_80', 'upper_80',
    'confidence_level', 'forecast_generated_date', 'forecast_horizon_weeks',
    'model_type', 'model_version', 'data_quality_note', 'forecast_generated_at'
]

df_forecast_output = df_forecast_output[output_cols]

print(f"✅ Prepared {len(df_forecast_output)} weeks of forecasts")

print("\n📋 4-Week Forecast Summary:")
print(df_forecast_output[[
    'forecast_week_end_date', 'predicted_cases', 
    'predicted_cases_lower', 'predicted_cases_upper'
]].to_string(index=False))

print()


📈 PREPARING FORECAST OUTPUT

✅ Prepared 4 weeks of forecasts

📋 4-Week Forecast Summary:
forecast_week_end_date  predicted_cases  predicted_cases_lower  predicted_cases_upper
            2025-03-02              792                    560                   1025
            2025-03-09              828                    543                   1114
            2025-03-16              864                    525                   1204
            2025-03-23              900                    507                   1293



In [19]:
# ============================================
# STEP 8: MODEL EXPLAINABILITY
# ============================================

print("\n" + "="*60)
print("🔍 MODEL EXPLAINABILITY")
print("="*60 + "\n")

# Calculate trend direction
trend_direction = 'INCREASING' if slope > 0 else ('DECREASING' if slope < 0 else 'STABLE')
trend_strength = 'Strong' if abs(r_value**2) > 0.7 else ('Moderate' if abs(r_value**2) > 0.4 else 'Weak')

print("📊 Model Components:")
print(f"   - Model Type: Simple Linear Trend")
print(f"   - Trend Direction: {trend_direction}")
print(f"   - Trend Strength: {trend_strength} (R² = {r_value**2:.3f})")
print(f"   - Slope: {slope:+.2f} cases/week")
print(f"   - Average Change: {slope:+.1f} cases per week")
print()

# Calculate total trend change over forecast period
total_change_forecast = slope * 4  # 4 weeks
print("📈 Forecast Explanation:")
print(f"   - Expected trend over 4 weeks: {total_change_forecast:+.0f} cases")
print(f"   - Week 1 prediction: {df_forecast_output.iloc[0]['predicted_cases']} cases")
print(f"   - Week 4 prediction: {df_forecast_output.iloc[-1]['predicted_cases']} cases")
print(f"   - Total predicted (4 weeks): {df_forecast_output['predicted_cases'].sum()} cases")
print()

print("⚠️ Uncertainty Analysis:")
print(f"   - Historical std dev: {std_cases:.0f} cases")
print(f"   - 95% confidence interval width (avg): {(df_forecast_output['predicted_cases_upper'] - df_forecast_output['predicted_cases_lower']).mean():.0f} cases")
print(f"   - Uncertainty increases with time (widening intervals)")
print()

print("📌 Model Limitations:")
print("   ⚠️ Only 3 weeks of historical data")
print("   ⚠️ No seasonal patterns detectable")
print("   ⚠️ Linear extrapolation may not capture complex dynamics")
print("   ⚠️ Use for demonstration and planning purposes only")
print()

# Create explainability summary
model_explanation = {
    'model_type': 'Linear Trend',
    'trend_direction': trend_direction,
    'trend_strength': trend_strength,
    'slope_cases_per_week': float(slope),
    'r_squared': float(r_value**2),
    'historical_mean': float(mean_cases),
    'historical_std': float(std_cases),
    'forecast_horizon_weeks': 4,
    'total_predicted_cases': int(df_forecast_output['predicted_cases'].sum()),
    'data_quality': 'Limited (3 weeks)',
    'recommended_use': 'Demonstration only - not for operational decisions',
    'created_at': datetime.now()
}

print("✅ Model explainability documented")
print()


🔍 MODEL EXPLAINABILITY

📊 Model Components:
   - Model Type: Simple Linear Trend
   - Trend Direction: INCREASING
   - Trend Strength: Weak (R² = 0.156)
   - Slope: +36.00 cases/week
   - Average Change: +36.0 cases per week

📈 Forecast Explanation:
   - Expected trend over 4 weeks: +144 cases
   - Week 1 prediction: 792 cases
   - Week 4 prediction: 900 cases
   - Total predicted (4 weeks): 3384 cases

⚠️ Uncertainty Analysis:
   - Historical std dev: 91 cases
   - 95% confidence interval width (avg): 625 cases
   - Uncertainty increases with time (widening intervals)

📌 Model Limitations:
   ⚠️ Only 3 weeks of historical data
   ⚠️ No seasonal patterns detectable
   ⚠️ Linear extrapolation may not capture complex dynamics
   ⚠️ Use for demonstration and planning purposes only

✅ Model explainability documented



In [22]:
# ============================================
# STEP 9: MODEL PERFORMANCE SUMMARY
# ============================================

print("\n" + "="*60)
print("📊 MODEL PERFORMANCE SUMMARY")
print("="*60 + "\n")

# Calculate performance metrics on training data
print("Calculating model performance metrics...")

from sklearn.metrics import mean_squared_error, mean_absolute_error

y_true = df_forecast_input['y'].values
y_pred = [intercept + slope * i for i in range(len(df_forecast_input))]

# Calculate metrics
rmse = mean_squared_error(y_true, y_pred, squared=False)
mae = mean_absolute_error(y_true, y_pred)
mape = (abs(y_true - y_pred) / y_true).mean() * 100
r_squared = r_value**2

print(f"   RMSE: {rmse:.2f}")
print(f"   MAE: {mae:.2f}")
print(f"   MAPE: {mape:.2f}%")
print(f"   R²: {r_squared:.3f}")
print()

# Create comprehensive performance record
df_model_performance = pd.DataFrame([{
    'model_id': 'linear_trend_continental_v1_mvp',
    'model_type': 'Linear Trend',
    'model_version': '1.0_mvp',
    'training_date': datetime.now().date(),
    'training_records': len(df_forecast_input),
    'training_period_weeks': len(df_forecast_input),
    'forecast_horizon_weeks': 4,
    
    # Performance metrics
    'rmse': float(rmse),
    'mae': float(mae),
    'mape': float(mape),
    'r_squared': float(r_squared),
    
    # Model parameters
    'slope': float(slope),
    'intercept': float(intercept),
    'trend_direction': trend_direction,
    
    # Prediction statistics
    'avg_predicted_cases': int(df_forecast_output['predicted_cases'].mean()),
    'total_predicted_cases': int(df_forecast_output['predicted_cases'].sum()),
    'min_predicted': int(df_forecast_output['predicted_cases'].min()),
    'max_predicted': int(df_forecast_output['predicted_cases'].max()),
    
    # Uncertainty metrics
    'avg_prediction_interval_width': int((df_forecast_output['predicted_cases_upper'] - df_forecast_output['predicted_cases_lower']).mean()),
    'confidence_level': 95,
    
    # Data quality indicators
    'data_quality': 'Limited',
    'data_quality_note': 'Only 3 weeks of historical data - insufficient for robust forecasting',
    'seasonality_detected': False,
    'anomalies_in_training': 0,
    
    # Operational metadata
    'model_purpose': 'MVP Demonstration',
    'recommended_use': 'Planning scenarios only - not for operational decisions',
    'next_retrain_recommended': (datetime.now() + pd.Timedelta(weeks=4)).date(),
    'created_at': datetime.now(),
    'created_by': 'Cholera CDR MVP Pipeline'
}])

print("✅ Model performance record created")

print("\n📊 Key Performance Indicators:")
print(f"   Model ID: {df_model_performance.iloc[0]['model_id']}")
print(f"   Training Records: {df_model_performance.iloc[0]['training_records']} weeks")
print(f"   RMSE: {df_model_performance.iloc[0]['rmse']:.2f} cases")
print(f"   MAE: {df_model_performance.iloc[0]['mae']:.2f} cases")
print(f"   MAPE: {df_model_performance.iloc[0]['mape']:.2f}%")
print(f"   R²: {df_model_performance.iloc[0]['r_squared']:.3f}")
print(f"   Avg Predicted: {df_model_performance.iloc[0]['avg_predicted_cases']} cases/week")
print(f"   Trend: {df_model_performance.iloc[0]['trend_direction']}")

print("\n📋 Full Performance Record:")
# Display selected columns
display_cols = [
    'model_id', 'training_records', 'rmse', 'mae', 'mape', 'r_squared',
    'avg_predicted_cases', 'trend_direction', 'data_quality'
]
print(df_model_performance[display_cols].to_string(index=False))

print()


📊 MODEL PERFORMANCE SUMMARY

Calculating model performance metrics...
   RMSE: 68.35
   MAE: 64.44
   MAPE: 9.21%
   R²: 0.156

✅ Model performance record created

📊 Key Performance Indicators:
   Model ID: linear_trend_continental_v1_mvp
   Training Records: 3 weeks
   RMSE: 68.35 cases
   MAE: 64.44 cases
   MAPE: 9.21%
   R²: 0.156
   Avg Predicted: 846 cases/week
   Trend: INCREASING

📋 Full Performance Record:
                       model_id  training_records      rmse       mae     mape  r_squared  avg_predicted_cases trend_direction data_quality
linear_trend_continental_v1_mvp                 3 68.353656 64.444444 9.208437   0.156063                  846      INCREASING      Limited



C:\Python312\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [25]:
# ============================================
# DEFINE PATHS (if not already defined)
# ============================================

from pathlib import Path

# Check if GOLD_PATH is defined, if not define it
try:
    GOLD_PATH
except NameError:
    if IS_FABRIC:
        GOLD_PATH = "/lakehouse/default/Tables/gold"
    else:
        GOLD_PATH = r"D:\Projects\cholera-cdr-mvp\data\gold_tables"
    
    print(f"✅ GOLD_PATH set to: {GOLD_PATH}")

✅ GOLD_PATH set to: D:\Projects\cholera-cdr-mvp\data\gold_tables


In [26]:
# ============================================
# SAVE TO GOLD LAYER
# ============================================

print("\n" + "="*60)
print("💾 SAVING TO GOLD LAYER")
print("="*60 + "\n")

gold_path = Path(GOLD_PATH)

# Save forecast
df_forecast_output.to_parquet(gold_path / "ml_forecast_cases_4wk.parquet", index=False)
print(f"✅ Saved ml_forecast_cases_4wk.parquet ({len(df_forecast_output)} rows)")

# Save model performance
df_model_performance.to_parquet(gold_path / "ml_model_performance.parquet", index=False)
print(f"✅ Saved ml_model_performance.parquet ({len(df_model_performance)} rows)")

# Save model explanation (optional)
df_explanation = pd.DataFrame([model_explanation])
df_explanation.to_parquet(gold_path / "ml_model_explanation.parquet", index=False)
print(f"✅ Saved ml_model_explanation.parquet (1 row)")

print(f"\n✅ All forecast outputs saved to: {GOLD_PATH}")

print("\n📦 Saved Files:")
print("   1. ml_forecast_cases_4wk.parquet - 4-week predictions")
print("   2. ml_model_performance.parquet - Model metrics")
print("   3. ml_model_explanation.parquet - Model interpretation")

print("\n" + "="*60)
print("✅ ML FORECASTING COMPLETE!")
print("="*60)


💾 SAVING TO GOLD LAYER

✅ Saved ml_forecast_cases_4wk.parquet (4 rows)
✅ Saved ml_model_performance.parquet (1 rows)
✅ Saved ml_model_explanation.parquet (1 row)

✅ All forecast outputs saved to: D:\Projects\cholera-cdr-mvp\data\gold_tables

📦 Saved Files:
   1. ml_forecast_cases_4wk.parquet - 4-week predictions
   2. ml_model_performance.parquet - Model metrics
   3. ml_model_explanation.parquet - Model interpretation

✅ ML FORECASTING COMPLETE!


## Validation & Testing

In [28]:
# ============================================
# VALIDATION & TESTING
# ============================================

print("\n" + "="*60)
print("🔍 VALIDATION & TESTING")
print("="*60 + "\n")

print("Running validation checks...\n")

FORECAST_WEEKS = 4  # Define expected forecast weeks

# Test 1: Forecast generated
assert len(df_forecast_output) == FORECAST_WEEKS, f"Expected {FORECAST_WEEKS} forecasts, got {len(df_forecast_output)}"
print(f"✅ Generated {FORECAST_WEEKS}-week forecast")

# Test 2: Predictions are non-negative
assert (df_forecast_output['predicted_cases'] >= 0).all(), "Negative predictions found"
assert (df_forecast_output['predicted_cases_lower'] >= 0).all(), "Negative lower bounds found"
print(f"✅ All predictions are non-negative")

# Test 3: Uncertainty intervals valid
assert (df_forecast_output['predicted_cases_upper'] >= df_forecast_output['predicted_cases']).all(), "Upper bound < prediction"
assert (df_forecast_output['predicted_cases'] >= df_forecast_output['predicted_cases_lower']).all(), "Prediction < lower bound"
print(f"✅ Uncertainty intervals are valid")

# Test 4: Model performance metrics exist
assert 'rmse' in df_model_performance.columns, "RMSE not in model performance"
assert 'mae' in df_model_performance.columns, "MAE not in model performance"
assert df_model_performance['rmse'].iloc[0] > 0, "RMSE not calculated"
assert df_model_performance['mae'].iloc[0] > 0, "MAE not calculated"
print(f"✅ Model performance metrics calculated")

# Test 5: Files were saved
import os
gold_path = Path(GOLD_PATH)
assert (gold_path / "ml_forecast_cases_4wk.parquet").exists(), "Forecast file not saved"
assert (gold_path / "ml_model_performance.parquet").exists(), "Performance file not saved"
assert (gold_path / "ml_model_explanation.parquet").exists(), "Explanation file not saved"
print(f"✅ All output files saved successfully")

# Test 6: Data quality checks
assert df_forecast_output['model_type'].iloc[0] == 'Linear Trend', "Model type incorrect"
assert df_forecast_output['confidence_level'].iloc[0] == 95, "Confidence level incorrect"
print(f"✅ Metadata validation passed")

# Final Summary
print("\n" + "="*60)
print("📊 FORECAST VALIDATION SUMMARY")
print("="*60)
print(f"\n📅 Forecast Period:")
print(f"   Start: {df_forecast_output['forecast_week_end_date'].min().date()}")
print(f"   End: {df_forecast_output['forecast_week_end_date'].max().date()}")
print(f"   Weeks: {len(df_forecast_output)}")

print(f"\n📈 Predictions:")
print(f"   Total predicted cases (4 weeks): {df_forecast_output['predicted_cases'].sum():,}")
print(f"   Average weekly prediction: {df_forecast_output['predicted_cases'].mean():.0f}")
print(f"   Minimum: {df_forecast_output['predicted_cases'].min():,} cases")
print(f"   Maximum: {df_forecast_output['predicted_cases'].max():,} cases")

print(f"\n🎯 Uncertainty:")
avg_interval = (df_forecast_output['predicted_cases_upper'] - df_forecast_output['predicted_cases_lower']).mean()
print(f"   Average 95% CI width: {avg_interval:.0f} cases")
print(f"   Confidence level: {df_forecast_output['confidence_level'].iloc[0]}%")

print(f"\n📊 Model Performance:")
print(f"   RMSE: {df_model_performance['rmse'].iloc[0]:.2f} cases")
print(f"   MAE: {df_model_performance['mae'].iloc[0]:.2f} cases")
print(f"   MAPE: {df_model_performance['mape'].iloc[0]:.2f}%")
print(f"   R²: {df_model_performance['r_squared'].iloc[0]:.3f}")

print(f"\n💾 Output Files:")
print(f"   1. ml_forecast_cases_4wk.parquet ({len(df_forecast_output)} rows)")
print(f"   2. ml_model_performance.parquet ({len(df_model_performance)} rows)")
print(f"   3. ml_model_explanation.parquet (1 row)")

print("\n" + "="*60)
print("✅ ALL VALIDATION CHECKS PASSED!")
print("="*60)
print("\n🎉 Notebook 05: ML Forecasting - COMPLETE!")
print("="*60)


🔍 VALIDATION & TESTING

Running validation checks...

✅ Generated 4-week forecast
✅ All predictions are non-negative
✅ Uncertainty intervals are valid
✅ Model performance metrics calculated
✅ All output files saved successfully
✅ Metadata validation passed

📊 FORECAST VALIDATION SUMMARY

📅 Forecast Period:
   Start: 2025-03-02
   End: 2025-03-23
   Weeks: 4

📈 Predictions:
   Total predicted cases (4 weeks): 3,384
   Average weekly prediction: 846
   Minimum: 792 cases
   Maximum: 900 cases

🎯 Uncertainty:
   Average 95% CI width: 625 cases
   Confidence level: 95%

📊 Model Performance:
   RMSE: 68.35 cases
   MAE: 64.44 cases
   MAPE: 9.21%
   R²: 0.156

💾 Output Files:
   1. ml_forecast_cases_4wk.parquet (4 rows)
   2. ml_model_performance.parquet (1 rows)
   3. ml_model_explanation.parquet (1 row)

✅ ALL VALIDATION CHECKS PASSED!

🎉 Notebook 05: ML Forecasting - COMPLETE!


## Next Steps

1. **Review forecast results** above
2. **Monitor forecast accuracy** as actual data comes in
3. **Retrain model** weekly with new data
4. **Use forecasts** in Power BI dashboard

## Outputs Created

- `gold.forecast_cases_4wk` - 4-week ahead forecasts with 95% prediction intervals
- `gold.model_performance` - Model evaluation metrics and metadata

**Pipeline Complete!** 🎉 All 5 notebooks executed successfully.